# OrbitGNN — Real TLE Benchmark Demo

End-to-end walkthrough using the **TLE Observation Benchmark Dataset**.

**Dataset**: https://github.com/dpshorten/TLE_observation_benchmark_dataset
**Code**: https://github.com/keshavgujrathi/OrbitGNN (branch: real-tle-pipeline)


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np, torch, math, pathlib, datetime, csv, warnings
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
DATASET_PATH = '../../TLE_observation_benchmark_dataset-main'
RESULTS_DIR  = pathlib.Path('../results/validation')
OUT_DIR      = pathlib.Path('../results')
from dataset import (load_real_benchmark_dataset, compute_residual_sequences,
                     build_orbital_neighbor_graph, fit_per_satellite_scaler, apply_per_satellite_scaler)
from model import OrbitGNN, anomaly_score
from physics import estimate_delta_v, R_EARTH
from train import make_windows, chronological_split, best_f1_threshold, roc_auc, pr_auc
SHELL_NAMES = {0:'GEO', 1:'SSO/Polar', 2:'LEO-66°'}
SHELL_COLS  = {0:'#FFD700', 1:'#4169E1', 2:'#32CD32'}
print('Setup complete. Dataset exists:', os.path.exists(DATASET_PATH))

## 1. Load Real TLE Benchmark Dataset

In [ ]:
result = load_real_benchmark_dataset(DATASET_PATH, start_date='2020-01-01',
    end_date='2022-01-01', dt_hours=24.0, max_tle_gap_hours=48.0, maneuver_tolerance_hours=24.0)
eo=result['elements_obs']; labels=result['labels']; sid=result['shell_id']
timestamps=result['timestamps']; sat_names=result['sat_names']
vm=result['valid_mask']; man_ev=result['maneuver_events']
dts=result['dt_seconds_grid']; meta=result['metadata']
S=meta['n_satellites']; T=meta['n_grid_steps']
print(f'Satellites: {S}  Grid steps: {T}  Period: {timestamps[0].date()} to {timestamps[-1].date()}')
print(f'Total manoeuvres: {meta["total_maneuver_events"]}')
print()
print(f'{"Satellite":<14} {"Shell":<12} {"Alt(km)":>8} {"Inc(deg)":>8} {"Manoeuvres":>11}')
print('-'*58)
for k,name in enumerate(sat_names):
    a=float(np.nanmean(eo[k,:,0])); alt=a-R_EARTH
    inc=float(np.degrees(np.nanmean(eo[k,:,2])))
    sh=SHELL_NAMES.get(int(sid[k]),'Other')
    print(f'{name:<14} {sh:<12} {alt:>8.0f} {inc:>8.1f} {len(man_ev.get(name,[])):>11}')

## 2. Physics Residuals Plot

In [ ]:
res = compute_residual_sequences(eo, dts)                # (S, T-1, 6)
vrm = vm[:, :-1] & vm[:, 1:]                            # valid residual mask (S, T-1)
T_train = int(0.60 * (T - 1))
sat_mean, sat_scale = fit_per_satellite_scaler(res, train_end=T_train)
ts_arr = np.array([t.replace(tzinfo=None) for t in timestamps], dtype='datetime64[s]')
ts_mid = ts_arr[:-1]
fig, axes = plt.subplots(S, 2, figsize=(15, 2.4*S), sharex=True)
for k,name in enumerate(sat_names):
    for col,(feat,fl,unit) in enumerate([(0,'Delta_a','km'),(4,'Delta_M','rad')]):
        ax=axes[k,col]; vals=res[k,:,feat]; mask=vrm[k,:]
        ax.plot(ts_mid[mask],vals[mask],color=SHELL_COLS.get(int(sid[k]),'grey'),lw=0.7,alpha=0.8)
        ax.axhline(0,color='k',lw=0.4)
        for mev in man_ev.get(name,[]):
            ax.axvline(np.datetime64(mev.strftime('%Y-%m-%dT%H:%M:%S')),color='red',lw=1.0,alpha=0.6,ls='--')
        if col==0: ax.set_ylabel(name,fontsize=7,rotation=0,ha='right',labelpad=55)
        ax.set_title(f'{fl} [{unit}]',fontsize=8); ax.tick_params(labelsize=7)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.suptitle('Physics Residuals -- red dashed = manoeuvre timestamps',fontsize=11,y=1.01)
plt.tight_layout()
out = OUT_DIR/'demo_residuals.png'
plt.savefig(out,dpi=100,bbox_inches='tight'); plt.close()
print(f'Saved: {out}')

## 3. Orbital Graph

In [ ]:
snap = eo[:,T//2,:]
adj_np = build_orbital_neighbor_graph(snap,sid,k_neighbors=3,cross_shell=False)
print(f'Graph: {S} nodes, {int(adj_np.sum()/2)} undirected edges')
try:
    import networkx as nx
    G=nx.from_numpy_array(adj_np)
    G=nx.relabel_nodes(G,{i:n for i,n in enumerate(sat_names)})
    pos={}
    for k,name in enumerate(sat_names):
        sh=int(sid[k]); peers=[n for n,s in enumerate(sid) if int(s)==sh]
        pos[name]=(sh*4.0, peers.index(k)-len(peers)/2)
    fig,ax=plt.subplots(figsize=(10,5))
    nx.draw_networkx(G,pos=pos,ax=ax,
        node_color=[SHELL_COLS.get(int(sid[i]),'grey') for i in range(S)],
        node_size=900,font_size=8,edge_color='#444',width=2.0,arrows=False)
    patches=[mpatches.Patch(color=c,label=f'Shell {s}: {SHELL_NAMES[s]}') for s,c in SHELL_COLS.items()]
    ax.legend(handles=patches); ax.set_title('OrbitGNN Orbital Graph'); ax.axis('off')
    plt.tight_layout()
    out=OUT_DIR/'demo_graph.png'; plt.savefig(out,dpi=100,bbox_inches='tight'); plt.close()
    print(f'Saved: {out}')
except ImportError:
    print('pip install networkx for graph plot')

## 4. Load Model and Score Test Window

In [ ]:
res_normed = apply_per_satellite_scaler(res, sat_mean, sat_scale)
WINDOW=8
X,Y,L,T_idx = make_windows(res_normed, vrm, labels, window=WINDOW)
n_windows=len(X); train_end,val_end=chronological_split(n_windows)
print(f'Windows: {n_windows}  train={train_end}  val={val_end}  test={n_windows-val_end}')
X_np=np.array(X); Xn=X_np.copy()
for k in range(S):
    tr=X_np[:train_end,k].reshape(-1,6); mu=tr.mean(0); sg=tr.std(0)+1e-8
    Xn[:,k]=((X_np[:,k]-mu)/sg).clip(-10,10)
MODEL_PATH=pathlib.Path('../results/best_model.pt')
model=OrbitGNN(in_dim=6,d_model=64,n_heads=4,n_layers=2)
if MODEL_PATH.exists():
    ckpt=torch.load(MODEL_PATH,map_location='cpu',weights_only=False)
    model.load_state_dict(ckpt.get('state_dict',ckpt))
    print(f'Loaded {MODEL_PATH}')
else:
    print(f'WARNING: {MODEL_PATH} not found, using random weights')
model.eval()
adj_t=torch.tensor(adj_np,dtype=torch.float32)
Xn_test=Xn[val_end:]; Y_test=np.array(Y[val_end:]); L_test=np.array(L[val_end:])
test_ts=timestamps[val_end:]
per_sat_all=[]
with torch.no_grad():
    for i in range(len(Xn_test)):
        x_t=torch.tensor(Xn_test[i],dtype=torch.float32)
        y_t=torch.tensor(Y_test[i],dtype=torch.float32)
        sf,pf,_=model(x_t,adj_t); mp,sp=model.mc_dropout_forecast(x_t,adj_t,n_samples=15)
        per_sat_all.append(anomaly_score(y_t,sf,pf,sp).numpy())
per_sat_arr=np.array(per_sat_all)
scores_flat=per_sat_arr.mean(axis=1); labels_flat=L_test.any(axis=1).astype(float)
roc_val=roc_auc(labels_flat,scores_flat); pr_val=pr_auc(labels_flat,scores_flat)
thr=best_f1_threshold(labels_flat,scores_flat)
print(f'ROC-AUC={roc_val:.4f}  PR-AUC={pr_val:.4f}  threshold={thr:.3f}')

## 5. Anomaly Timeline

In [ ]:
TOL_72H=72*3600.0
test_ts_np=np.array([t.replace(tzinfo=None) for t in test_ts],dtype='datetime64[s]')
ts0=test_ts[0].replace(tzinfo=None); ts1=test_ts[-1].replace(tzinfo=None)
n_test_man=[len([m for m in man_ev.get(n,[]) if ts0<=m.replace(tzinfo=None)<=ts1]) for n in sat_names]
top3=np.argsort(n_test_man)[::-1][:3]
fig,axes=plt.subplots(len(top3),1,figsize=(15,4*len(top3)),sharex=True)
if len(top3)==1: axes=[axes]
for ax,k in zip(axes,top3):
    name=sat_names[k]; sc_k=per_sat_arr[:,k]; color=SHELL_COLS.get(int(sid[k]),'grey')
    ax.plot(test_ts_np[:len(sc_k)],sc_k,color=color,lw=0.8)
    ax.axhline(thr,color='orange',lw=1.2,ls='--')
    ax.fill_between(test_ts_np[:len(sc_k)],0,sc_k,where=(sc_k>thr),color='orange',alpha=0.25)
    for mev in man_ev.get(name,[]):
        mt_s=mev.replace(tzinfo=None)
        if not(ts0<=mt_s<=ts1): continue
        mt_np=np.datetime64(mt_s.strftime('%Y-%m-%dT%H:%M:%S'))
        alarms=test_ts_np[:len(sc_k)][sc_k>thr]
        det=(len(alarms)>0 and abs((alarms-mt_np).astype('timedelta64[s]').astype(float)).min()<=TOL_72H)
        ax.axvline(mt_np,color='#00AA00' if det else 'red',lw=1.5,alpha=0.8,ls='-' if det else '--')
    ax.set_ylabel(f'{name}\nScore',fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m')); ax.tick_params(labelsize=8)
    custom=[Line2D([0],[0],color=color,lw=1.5,label='Score'),
            Line2D([0],[0],color='orange',lw=1.5,ls='--',label=f'Threshold'),
            Line2D([0],[0],color='#00AA00',lw=1.5,label='Detected'),
            Line2D([0],[0],color='red',lw=1.5,ls='--',label='Missed')]
    ax.legend(handles=custom,fontsize=8,loc='upper right')
axes[-1].set_xlabel('Date')
fig.suptitle('OrbitGNN Anomaly Score Timeline',fontsize=12,y=1.01)
plt.tight_layout()
out=OUT_DIR/'demo_timeline.png'; plt.savefig(out,dpi=100,bbox_inches='tight'); plt.close()
print(f'Saved: {out}')

## 6. Ablation Comparison

In [ ]:
abl_file=RESULTS_DIR/'ablation_results.csv'
if abl_file.exists():
    configs=['physics_only','transformer_only','gnn_only','full']
    labs=['Physics\nOnly','Physics+\nTransformer','Physics+\nGNN','Full\nOrbitGNN']
    md={c:{'roc':[],'pr':[],'f1':[]} for c in configs}
    with open(abl_file) as f:
        for row in csv.DictReader(f):
            c=row['ablation']
            if c in md: md[c]['roc'].append(float(row['roc_auc'])); md[c]['pr'].append(float(row['pr_auc'])); md[c]['f1'].append(float(row['f1']))
    fig,axes=plt.subplots(1,3,figsize=(13,4))
    colors_abl=['#BBBBBB','#4169E1','#32CD32','#FF6600']
    for ax,(metric,ylabel) in zip(axes,[('roc','ROC-AUC'),('pr','PR-AUC'),('f1','F1')]):
        vals=[np.mean(md[c][metric]) for c in configs]; errs=[np.std(md[c][metric]) for c in configs]
        bars=ax.bar(labs,vals,yerr=errs,color=colors_abl,capsize=5,edgecolor='k',linewidth=0.8)
        ax.set_title(ylabel,fontsize=11); ax.set_ylim(0,max(vals)*1.35); ax.tick_params(axis='x',rotation=15,labelsize=8)
        for bar,v in zip(bars,vals): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.003,f'{v:.3f}',ha='center',va='bottom',fontsize=8)
    fig.suptitle('OrbitGNN Ablation Study',fontsize=12); plt.tight_layout()
    out=OUT_DIR/'demo_ablation.png'; plt.savefig(out,dpi=100,bbox_inches='tight'); plt.close()
    print(f'Saved: {out}')
else:
    print('Run run_experiments.py first')

## 7. Delta-V Estimation and Summary

In [ ]:
from physics import estimate_delta_v
print(f'{"Satellite":<14} {"Manoeuvre date":<22} {"|Da|(km)":>9} {"DV(m/s)":>9}')
print('-'*58)
for k,name in enumerate(sat_names):
    for mev in man_ev.get(name,[]):
        mt_s=mev.replace(tzinfo=None)
        if not(ts0<=mt_s<=ts1): continue
        diffs=[abs((t.replace(tzinfo=None)-mt_s).total_seconds()) for t in test_ts]
        if not diffs: continue
        w_idx=int(np.argmin(diffs)); t_abs=T_idx[val_end+w_idx] if (val_end+w_idx)<len(T_idx) else -1
        if t_abs<0 or t_abs>=res.shape[1]: continue
        da=abs(res[k,t_abs,0]); a=eo[k,t_abs,0]
        if a>0 and da>0.001:
            dv=estimate_delta_v(da,a)
            print(f'{name:<14} {mev.strftime("%Y-%m-%d %H:%M"):<22} {da:>9.4f} {dv:>9.3f}')
print()
print('Summary of saved results:')
if (RESULTS_DIR/'bootstrap_ci.csv').exists():
    with open(RESULTS_DIR/'bootstrap_ci.csv') as f:
        for row in csv.DictReader(f):
            print(f'  {row["metric"]:10s}: {float(row["estimate"]):.4f} (95% CI: [{float(row["ci_lower"]):.4f}, {float(row["ci_upper"]):.4f}])')
if (RESULTS_DIR/'statistical_tests.csv').exists():
    print()
    print('Statistical tests (OrbitGNN vs baselines, t-test):')
    with open(RESULTS_DIR/'statistical_tests.csv') as f:
        for row in csv.DictReader(f):
            if row['metric']=='roc_auc':
                sig='YES' if row['significant']=='True' else 'NO'
                print(f'  vs {row["baseline"]:18s}: delta={float(row["delta"]):+.4f} p={float(row["p_value"]):.4f} sig={sig}')